# Error angle and distance to bead data from smooth data

In [3]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Save error angle and distance data csv

In [4]:
#Input dir- folder path of smooth csv, output dir- where the chase metric csv should be saved
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Error angle functions
def compute_error_angle_center(bead, head, center):
    v_heading = head - center
    v_bead    = head - bead
    dot = np.einsum("ij,ij->i", v_heading, v_bead)
    nh = np.linalg.norm(v_heading, axis=1)
    nb = np.linalg.norm(v_bead, axis=1)
    cos = np.clip(dot / (nh * nb), -1, 1)
    return np.degrees(np.arccos(cos))

def compute_error_angle_head(bead, head):
    v_bead = bead - head
    forward = np.gradient(head, axis=0)
    dot = np.einsum("ij,ij->i", forward, v_bead)
    nf = np.linalg.norm(forward, axis=1)
    nb = np.linalg.norm(v_bead, axis=1)
    cos = np.clip(dot / (nf * nb), -1, 1)
    return np.degrees(np.arccos(cos))

# Find smooth files
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))
if not csv_files:
    raise FileNotFoundError("No SMOOTH CSV files found.")


for path in csv_files:
    fname = os.path.basename(path)
    print("Processing:", fname)

    df = pd.read_csv(path)

    # Use frame directly from CSV
    frames = df["frame"].values

    bead   = df[["pt1_X","pt1_Y","pt1_Z"]].values
    head   = df[["pt2_X","pt2_Y","pt2_Z"]].values
    center = df[["center_X","center_Y","center_Z"]].values

    # Distances
    dist_center = np.linalg.norm(bead - center, axis=1)
    dist_head   = np.linalg.norm(bead - head, axis=1)

    # Error angles
    err_center = compute_error_angle_center(bead, head, center)
    err_head   = compute_error_angle_head(bead, head)

    # Save CSV
    out_df = pd.DataFrame({
        "frame": frames,
        "dist_center_m": dist_center,
        "dist_head_m": dist_head,
        "error_angle_center_deg": err_center,
        "error_angle_head_deg": err_head
    })

    out_name = fname.replace("_SMOOTH.csv", "_CHASE_METRICS.csv")
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

    print("Saved:", out_name)

print("\n Chase metrics CSVs saved.")

Processing: Trial2_200rpmxyzpts_SMOOTH.csv
Saved: Trial2_200rpmxyzpts_CHASE_METRICS.csv
Processing: Trial3_180rpmxyzpts_SMOOTH.csv
Saved: Trial3_180rpmxyzpts_CHASE_METRICS.csv
Processing: Trial4_400rpmxyzpts_SMOOTH.csv
Saved: Trial4_400rpmxyzpts_CHASE_METRICS.csv
Processing: Trial5_180rpmxyzpts_SMOOTH.csv
Saved: Trial5_180rpmxyzpts_CHASE_METRICS.csv

 Chase metrics CSVs saved.
